In [2]:
!pip install praw


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
# plan:
# go on reddit maybe through API and look up "{product_name} + reviews" 
# scrape the top 5 posts for texts and comments and then save into a text file called like "{product}_reddit_user_reviews"
import os
import time
import requests

PRODUCT_NAME = "AG1 Powder"
TOP_N_POSTS = 5
MAX_COMMENTS_PER_POST = 50

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; reddit-scraper/1.0)"}


# %%
def main():
    posts = search_reddit(PRODUCT_NAME, TOP_N_POSTS)
    write_to_file(posts, PRODUCT_NAME)
    print("DONE!")


def search_reddit(product_name, top_n):
    query = f"{product_name} reviews"
    print(f"Searching Reddit for: '{query}'")

    url = "https://www.reddit.com/search.json"
    params = {
        "q": query,
        "sort": "relevance",
        "limit": top_n,
        "type": "link",
    }

    response = requests.get(url, headers=HEADERS, params=params)
    response.raise_for_status()
    data = response.json()

    posts_raw = data["data"]["children"]
    results = []

    for i, child in enumerate(posts_raw, start=1):
        post = child["data"]
        print(f"\nFetching post {i}/{top_n}: {post['title'][:80]}")

        comments = fetch_comments(post["permalink"])

        results.append(
            {
                "rank": i,
                "title": post["title"],
                "subreddit": post["subreddit"],
                "author": post.get("author", "[deleted]"),
                "score": post["score"],
                "url": post.get("url", ""),
                "permalink": f"https://www.reddit.com{post['permalink']}",
                "selftext": post.get("selftext", "").strip(),
                "num_comments": post["num_comments"],
                "comments": comments,
            }
        )

        time.sleep(1)

    return results


def fetch_comments(permalink):
    url = f"https://www.reddit.com{permalink}.json"
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    data = response.json()

    comments = []
    if len(data) < 2:
        return comments

    comment_listing = data[1]["data"]["children"]

    for child in comment_listing[:MAX_COMMENTS_PER_POST]:
        if child["kind"] != "t1":
            continue
        c = child["data"]
        comments.append(
            {
                "author": c.get("author", "[deleted]"),
                "score": c.get("score", 0),
                "body": c.get("body", "").strip(),
            }
        )

    return comments


def write_to_file(posts, product_name):
    safe_name = product_name.replace(" ", "_").replace("/", "-")
    file_name = f"{safe_name}_reddit_user_reviews.txt"
    folder_path = "../../nlp_final/data/raw"
    file_path = os.path.join(folder_path, file_name)

    os.makedirs(folder_path, exist_ok=True)

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(f"Reddit User Reviews: {product_name}\n")
        f.write(f"Search query: '{product_name} reviews'\n")
        f.write(f"Top {len(posts)} posts\n")
        f.write("=" * 80 + "\n\n")

        for post in posts:
            f.write(f"POST #{post['rank']}\n")
            f.write(f"Title    : {post['title']}\n")
            f.write(f"Subreddit: r/{post['subreddit']}\n")
            f.write(f"Author   : u/{post['author']}\n")
            f.write(f"Score    : {post['score']}\n")
            f.write(f"Comments : {post['num_comments']}\n")
            f.write(f"Link     : {post['permalink']}\n")
            f.write("-" * 40 + "\n")

            if post["selftext"]:
                f.write("POST BODY:\n")
                f.write(post["selftext"] + "\n")
                f.write("-" * 40 + "\n")

            if post["comments"]:
                f.write(f"TOP {len(post['comments'])} COMMENTS:\n\n")
                for j, comment in enumerate(post["comments"], start=1):
                    f.write(f"  [{j}] u/{comment['author']} (score: {comment['score']})\n")
                    indented = "\n".join(
                        f"      {line}" for line in comment["body"].splitlines()
                    )
                    f.write(indented + "\n\n")
            else:
                f.write("No comments found.\n")

            f.write("=" * 80 + "\n\n")

    print(f"\nSaved to: {file_path}")


if __name__ == "__main__":
    main()

Searching Reddit for: 'AG1 Powder reviews'

Fetching post 1/5: Anyone tried microgreens powder? Trying to understand if it's actually different

Fetching post 2/5: How does AG1 Greens Powder Review Stack Up for Gut Health Needs?

Fetching post 3/5: Athletic Greens (AG1) worth it?

Fetching post 4/5: Thoughts on - ‘Huberman Husbands,’ ‘Bro Diets’ and the ‘Masculine’ Branding of F

Fetching post 5/5: Prodcast Roundup 2/2 - 2/24

Saved to: ../../nlp_final/data/raw/AG1_Powder_reddit_user_reviews.txt
DONE!
